# Publish New RTI Requests to Kafka

This notebook simulates the MEERA data flow:

1. Read `data/New_Requests.csv`
2. Take the **top 2 rows** (first 2 data records)
3. Publish each row as a JSON message to the Confluent Kafka topic `meera-new-requests`

The topic is configured with **1 minute retention** (`retention.ms=60000`), so messages expire after 60 seconds if not consumed.

**Prerequisites:** Kafka must be running and the topic created:
```bash
# Option A: Confluent Kafka via Docker
docker compose up -d
./scripts/create_topic.sh

# Option B: Local Kafka (no Docker)
./scripts/start_kafka_local.sh
./scripts/create_topic.sh
```

In [ ]:
import json
from pathlib import Path

import pandas as pd
from confluent_kafka import Producer
from confluent_kafka.admin import AdminClient, NewTopic

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CSV_PATH = PROJECT_ROOT / "data" / "New_Requests.csv"
KAFKA_BOOTSTRAP_SERVERS = "127.0.0.1:9092"
KAFKA_TOPIC = "meera-new-requests"
TOP_N_ROWS = 2
RETENTION_MS = 60_000  # 1 minute

print(f"Project root: {PROJECT_ROOT}")
print(f"CSV path: {CSV_PATH}")

In [ ]:
def ensure_topic_exists(topic: str, bootstrap_servers: str, retention_ms: int) -> None:
    admin = AdminClient({"bootstrap.servers": bootstrap_servers})
    metadata = admin.list_topics(timeout=10)
    if topic in metadata.topics:
        print(f"Topic '{topic}' already exists.")
        return

    futures = admin.create_topics(
        [
            NewTopic(
                topic,
                num_partitions=1,
                replication_factor=1,
                config={"retention.ms": str(retention_ms)},
            )
        ]
    )
    for name, future in futures.items():
        future.result()
        print(f"Created topic '{name}' with retention.ms={retention_ms}")


ensure_topic_exists(KAFKA_TOPIC, KAFKA_BOOTSTRAP_SERVERS, RETENTION_MS)

In [ ]:
df = pd.read_csv(CSV_PATH)
top_rows = df.head(TOP_N_ROWS)

print(f"Loaded {len(df)} rows from CSV; publishing top {len(top_rows)} rows.")
top_rows

In [ ]:
def delivery_report(err, msg):
    if err is not None:
        raise RuntimeError(f"Delivery failed for {msg.key()}: {err}")
    print(
        f"Published to {msg.topic()} [partition {msg.partition()}] "
        f"offset {msg.offset()} key={msg.key()!r}"
    )


producer = Producer({"bootstrap.servers": KAFKA_BOOTSTRAP_SERVERS})

for idx, row in top_rows.iterrows():
    payload = row.where(pd.notna(row), None).to_dict()
    message_key = str(payload.get("Registration No.", idx))
    producer.produce(
        topic=KAFKA_TOPIC,
        key=message_key,
        value=json.dumps(payload, ensure_ascii=False),
        callback=delivery_report,
    )

producer.flush()
print(f"Done. {len(top_rows)} messages are on topic '{KAFKA_TOPIC}'.")